# Show and Tell: Image Captioning Implementation
Modern implementation using TensorFlow 2.x and current best practices

In [ ]:
# Install required packages
!pip install tensorflow
!pip install nltk tqdm pandas matplotlib pycocotools
!pip install Pillow

In [ ]:
# Import necessary libraries
import tensorflow as tf
import numpy as np
import os
import pickle
import nltk
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16
nltk.download('punkt')
from google.colab import drive

In [ ]:
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Modern implementation of the Show and Tell model using Keras
class ShowAndTellModel(tf.keras.Model):
    def __init__(self, vocab_size, max_length, embedding_dim=512, units=512):
        super(ShowAndTellModel, self).__init__()
        
        # Load pretrained VGG16 without top layers
        self.cnn = VGG16(include_top=False, weights='imagenet')
        self.cnn.trainable = False
        
        # Feature extraction layers
        self.features_extract = tf.keras.Sequential([
            layers.GlobalAveragePooling2D(),
            layers.Dense(embedding_dim)
        ])
        
        # Caption generation layers
        self.embedding = layers.Embedding(vocab_size, embedding_dim)
        self.lstm = layers.LSTM(units, return_sequences=True, return_state=True)
        self.dense = layers.Dense(vocab_size)
        
    def call(self, inputs):
        image, captions = inputs
        
        # Extract image features
        features = self.cnn(image)
        features = self.features_extract(features)
        
        # Embed captions
        x = self.embedding(captions)
        
        # LSTM with attention
        output, state_h, state_c = self.lstm(x, initial_state=[features, features])
        
        # Generate word probabilities
        x = self.dense(output)
        
        return x

In [ ]:
# Modern data pipeline using tf.data
def create_dataset(image_paths, captions, batch_size=32):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, captions))
    dataset = dataset.shuffle(1000)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
# Training configuration using modern practices
class Config:
    def __init__(self):
        self.batch_size = 32
        self.embedding_dim = 512
        self.units = 512
        self.vocab_size = 5000
        self.max_length = 20
        self.learning_rate = 0.001
        
        # Use Colab paths
        self.train_image_dir = '/content/train2014/'
        self.val_image_dir = '/content/val2014/'
        self.checkpoint_path = '/content/drive/MyDrive/show_and_tell/checkpoints/'

config = Config()

In [ ]:
# Modern training loop with tf.keras
@tf.function
def train_step(model, optimizer, images, captions, loss_function):
    with tf.GradientTape() as tape:
        predictions = model([images, captions])
        loss = loss_function(captions, predictions)
        
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

In [ ]:
# Initialize model and optimizer
model = ShowAndTellModel(config.vocab_size, config.max_length,
                        config.embedding_dim, config.units)
optimizer = tf.keras.optimizers.Adam(config.learning_rate)
loss_function = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

# Checkpoint manager for saving models
ckpt = tf.train.Checkpoint(model=model, optimizer=optimizer)
ckpt_manager = tf.train.CheckpointManager(ckpt, config.checkpoint_path, max_to_keep=5)

In [ ]:
# Modern caption generation function
@tf.function
def generate_caption(model, image, tokenizer, max_length=20):
    attention_plot = np.zeros((max_length, attention_features_shape))
    
    hidden = model.reset_state(batch_size=1)
    temp_input = tf.expand_dims(load_image(image)[0], 0)
    img_tensor_val = model.cnn(temp_input)
    img_tensor_val = model.features_extract(img_tensor_val)
    
    dec_input = tf.expand_dims([tokenizer.word_index['<start>']], 0)
    result = []

    for i in range(max_length):
        predictions, hidden, attention_weights = model.decode_step(
            dec_input, img_tensor_val, hidden)
        
        predicted_id = tf.random.categorical(predictions, 1)[0][0].numpy()
        result.append(tokenizer.index_word[predicted_id])
        
        if tokenizer.index_word[predicted_id] == '<end>':
            return result
            
        dec_input = tf.expand_dims([predicted_id], 0)
    
    return result